# 04 Report and model diff

Renders the markdown report with its figures from the saved score table and diagnostics, shows it inline, and compares the AIFS-single and IFS score tables cell by cell against their confidence intervals.

## What this notebook does and does not show

Does: rebuild the `diag` dictionary exactly as `afv report` does, call `render_report`, display `report.md` and every figure under `results/.../figures`, then run `diff_tables` and `summarise` for `aifs-single` against `ifs`.

Does not: use the network, recompute any score, or support a general claim about either model. The closing cell lists what the whole pipeline cannot show.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from afv.config import load_config
from afv.dataset import read_dataset, read_manifest
from afv.diagnostics import qq
from afv.report import render_report
from afv.schema import ScoreTable

CONFIG = Path(os.environ.get("AFV_CONFIG", "configs/default.yaml"))
cfg = load_config(CONFIG)
out_dir = cfg.get_path("results_dir")
st = ScoreTable.from_parquet(out_dir / "scores.parquet")
matched = read_dataset(out_dir / "matched.parquet")
man = read_manifest(out_dir / "matched.parquet")
print(CONFIG, "->", cfg["name"])
print(len(st.rows), "score rows | dataset", st.dataset_hash, "| config", st.config_hash, "| created", st.created_utc)

## Build the diagnostics dictionary and render

Same shape as `cli.cmd_report`: the three CSVs from `afv diag` (None if absent) plus a fresh Q-Q table per model and variable at the primary interpolation.

In [ ]:
primary = cfg["interp"].get("primary", {})
diag = {}
for name in ("variance_ratio", "tail", "activity"):
    p = out_dir / f"diag_{name}.csv"
    diag[name] = pd.read_csv(p) if p.exists() else None
    print(f"{name:15s}", "loaded" if diag[name] is not None else "missing (run afv diag or notebook 03)")
diag["qq"] = [qq(matched, m, v, interp=primary.get(v)) for (m, v), _ in matched.groupby(["model", "variable"])]
print(len(diag["qq"]), "qq tables")

report_path = render_report(st, diag, out_dir, cfg, man)
print(report_path)

The report links its figures relative to `results/.../report.md`, so the images do not resolve when the markdown is shown from the notebook folder. The text is displayed first and the figures follow in a separate cell.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(report_path.read_text(encoding="utf-8")))

In [ ]:
from IPython.display import Image

figs = sorted((out_dir / "figures").glob("*.png"))
print(len(figs), "figures")
for p in figs:
    print(p.name)
    display(Image(filename=str(p)))

## AIFS-single against IFS

`diff_tables` joins the two per-model score tables on (variable, lead, stratum, metric), computes `delta = ifs - aifs-single`, flags whether the two 90 percent intervals overlap, and whether the second table is better (smaller error; smaller absolute bias). `summarise` counts the lead-variable cells in the `all` stratum. A cell counts as separated only when the intervals do not overlap, which is a conservative reading.

In [ ]:
from afv.diff import diff_tables, summarise

a = ScoreTable.from_parquet(out_dir / "scores_aifs-single.parquet")
b = ScoreTable.from_parquet(out_dir / "scores_ifs.parquet")
d = diff_tables(a, b, "aifs-single", "ifs")
print(summarise(d, "aifs-single", "ifs"))
d.to_csv(out_dir / "diff_aifs-single_vs_ifs.csv", index=False)
d[d["stratum"] == "all"].sort_values(["variable", "metric", "lead_h"]).round(3)

In [ ]:
# The same view by stratum, for the station_class and county breakdowns.
sep = d[(~d["ci_overlap"]) & (d["stratum"] != "all")]
print(len(sep), "stratified cells with separated intervals")
sep.sort_values(["stratum", "stratum_value", "variable", "metric", "lead_h"]).round(3)

## What this does not show

* One short window in one season. Nothing here speaks to extremes, to other seasons, or to year-round skill.
* 30 to 60 stations, with representativeness error at coastal and mountain sites: a point observation in a fjord or on a ridge is not what a 0.25 degree grid cell represents, and that error lands on both models alike.
* Raw model output, not the post-processed Yr product people actually use. Yr adds MEPS, statistical correction and human editing.
* Deterministic scores only, unless the lagged ensemble section in notebook 03 was run, and that ensemble mixes lead times.
* 0.25 degree global fields against point observations. The regional MEPS baseline at 2.5 km is Tier 1 and is not in this report.
* Nearest and bilinear interpolation give different numbers, which is why both are stored and the non-primary one is shown as its own stratum.
* The lapse correction uses a proxy orography (MEPS surface geopotential averaged to 0.25 degree), not the models' own surface height, which open data does not carry.
* The Frost quality accept list `{0, 1}` is a choice. Different accept codes change the sample; the raw `obs_qc` column is kept so the choice can be revisited.

The full list with references is in `docs/WHAT_THIS_DOES_NOT_SHOW.md`.